In [93]:
import matplotlib.pyplot as plt  
import numpy as np               
import pandas as pd              
from scipy import stats          
import geopandas as gpd
import seaborn as sns


# Data understanding

###  Leitura do dataset

In [94]:
listing = "dataset/listings.csv"
df = pd.read_csv(listing,)
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,48154,Precioso apartamento con wifi,219476,Antonio,LA SAIDIA,MORVEDRE,39.48375,-0.37502,Entire home/apt,83.0,3,197,2025-09-15,1.08,4,150,26,VT-41540-V
1,137143,PENTHOUSE ON FRONT LINE BEACH,670775,Maria De La Piedad,POBLATS DEL SUD,EL SALER,39.36335,-0.31932,Entire home/apt,390.0,10,1,2013-07-02,0.01,5,20,0,VT32745V
2,149715,1900 Style Valencian Beach Home for 10px,5947,Susana Barbara,POBLATS MARITIMS,CABANYAL-CANYAMELAR,39.46746,-0.32813,Entire home/apt,245.0,2,313,2025-09-15,1.81,1,287,41,ESFCTU000046025000580569000000000000000000VT-3...
3,165971,★ Architectural touch! ★,791187,Inés,EXTRAMURS,LA ROQUETA,39.46790,-0.38206,Entire home/apt,124.0,5,576,2025-09-03,3.34,8,106,46,VT-32757-V
4,182221,Apartments Calatrava City Valencia,1315567,Chiara,CAMINS AL GRAU,AIORA,39.46343,-0.34325,Entire home/apt,137.0,3,9,2025-03-17,0.07,1,176,1,VT-38755-V


### Remover o Id

In [95]:
df = df.drop("id", axis=1)

In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7844 entries, 0 to 7843
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   name                            7844 non-null   object 
 1   host_id                         7844 non-null   int64  
 2   host_name                       7838 non-null   object 
 3   neighbourhood_group             7844 non-null   object 
 4   neighbourhood                   7844 non-null   object 
 5   latitude                        7844 non-null   float64
 6   longitude                       7844 non-null   float64
 7   room_type                       7844 non-null   object 
 8   price                           6979 non-null   float64
 9   minimum_nights                  7844 non-null   int64  
 10  number_of_reviews               7844 non-null   int64  
 11  last_review                     6867 non-null   object 
 12  reviews_per_month               68

### descrição inicial

In [97]:
quantiva_discretas = ["number_of_reviews_ltm", "number_of_reviews", "minimum_nights", "calculated_host_listings_count"]
quantiva_continuas = ["latitude", "longitude", "price", "reviews_per_month", "availability_365"]
qualitativas_nominais = ["neighbourhood_group", "neighbourhood", "host_name","room_type"]

df["availability_365"].head(10)

0    150
1     20
2    287
3    106
4    176
5    161
6    160
7     88
8      1
9     45
Name: availability_365, dtype: int64

### Colocar as variáveis nominais como fatores

In [98]:
for col in qualitativas_nominais:
    df[col] = df[col].astype('category')
    
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7844 entries, 0 to 7843
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   name                            7844 non-null   object  
 1   host_id                         7844 non-null   int64   
 2   host_name                       7838 non-null   category
 3   neighbourhood_group             7844 non-null   category
 4   neighbourhood                   7844 non-null   category
 5   latitude                        7844 non-null   float64 
 6   longitude                       7844 non-null   float64 
 7   room_type                       7844 non-null   category
 8   price                           6979 non-null   float64 
 9   minimum_nights                  7844 non-null   int64   
 10  number_of_reviews               7844 non-null   int64   
 11  last_review                     6867 non-null   object  
 12  reviews_per_month   

In [99]:
def describe_quantative(tipo, excell ):
    desc = df[tipo].describe().round(3)

    desc.loc["amplitude"] = (df[tipo].max() - df[tipo].min()).round(3)
    desc.loc["IQR"] = (df[tipo].quantile(0.75) - df[tipo].quantile(0.25)).round(3)
    desc.loc["assimetria"] = df[tipo].skew().round(3)
    desc.loc["curtose"] = df[tipo].kurtosis().round(3)
    if excell == 1:
        desc.to_excel(f"desc_{tipo}.xlsx")
    display(desc)
    return
describe_quantative(quantiva_continuas, 0)

,latitude,longitude,price,reviews_per_month,availability_365
count,7844.000,7844.000,6979.000,6867.000,7844.000
mean,39.467,-0.364,164.113,1.695,185.951
std,0.019,0.023,732.639,1.666,120.086
min,39.280,-0.426,8.000,0.010,0.000
25%,39.461,-0.380,71.000,0.470,77.000
50%,39.469,-0.371,102.000,1.170,179.000
75%,39.475,-0.344,140.000,2.490,305.000
max,39.546,-0.276,40000.000,37.710,365.000
amplitude,0.266,0.150,39992.000,37.700,365.000
IQR,0.013,0.036,69.000,2.020,228.000


In [101]:
# função para descrever variáveis categóricas
def describe_categorical(df, variaveis, excell=0):
    resultado = []

    for var in variaveis:
        serie = df[var]
        
        n = serie.count()
        cardinalidade = serie.nunique(dropna=True)
        
        if not serie.mode().empty:
            moda = serie.mode().iloc[0]
            freq_moda = serie.value_counts().iloc[0]
            freq_rel_moda = round((freq_moda / n) * 100, 3) if n > 0 else 0
        else:
            moda = None
            freq_moda = None
            freq_rel_moda = None

        resultado.append({
            "Variável": var,
            "N": n,
            "Cardinalidade": cardinalidade,
            "Moda": moda,
            "Frequência da moda": freq_moda,
            "Frequência relativa da moda (%)": freq_rel_moda
        })
    resultado_df = pd.DataFrame(resultado)
    if excell:
        resultado_df.to_excel("desc_categoricas.xlsx", index=False)
    return pd.DataFrame(resultado)
describe_categorical(df,qualitativas_nominais )

,Variável,N,Cardinalidade,Moda,Frequência da moda,Frequência relativa da moda (%)
0,neighbourhood_group,7844,19,POBLATS MARITIMS,1471,18.753
1,neighbourhood,7844,85,CABANYAL-CANYAMELAR,750,9.561
2,host_name,7838,1764,Marta,124,1.582
3,room_type,7844,4,Entire home/apt,5782,73.712


In [ ]:

pie_chart_data = room.sort_values('Freq', ascending=False)
plt.figure(figsize=(10, 7))
colors = plt.cm.Set3(range(len(pie_chart_data)))
wedges, texts, autotexts = plt.pie(
    pie_chart_data['Freq'], 
    labels=pie_chart_data['Var1'], 
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    textprops={'fontsize': 12, 'weight': 'bold'}
)
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_fontsize(11)
plt.title("Quantidade de quartos, divididos por o tipo dos quartos", fontsize=14, weight='bold', pad=20)
plt.tight_layout()
plt.show()


In [103]:
# -------------------------------
# Scatter plots: price relationships
# -------------------------------

def pr_num_rev ():
    plt.figure()
    plt.scatter(df['price'], df['number_of_reviews'])
    plt.xlabel("Preço")
    plt.ylabel("Numero de reviews")
    plt.title("Numero de reviews VS Preço")
    plt.show()

def pr_rev_m ():
    plt.figure()
    plt.scatter(df['price'], df['reviews_per_month'])
    plt.xlabel("Preço")
    plt.ylabel("Reviews por mês")
    plt.title("Numero de reviews mensais VS Preço")
    plt.show()

def pr_rev_y ():
    plt.figure()
    plt.scatter(df['price'], df['number_of_reviews_ltm'])
    plt.xlabel("Preço")
    plt.ylabel("Reviews por ano")
    plt.title("Numero de reviews anuais VS Preço")
    plt.show()

def pr_ava ():
    plt.figure()
    plt.scatter(df['price'], df['availability_365'])
    plt.xlabel("Preço")
    plt.ylabel("Disponibilidade anual")
    plt.title("Disponibilidade anual VS Preço")
    plt.show()


In [102]:

# Load dataset

# -------------------------------
# Room type plots
# -------------------------------

room = df['room_type'].value_counts().reset_index()
room.columns = ['Var1', 'Freq']

# Pie chart
def pie_chart ():
    plt.figure()
    plt.pie(room['Freq'], labels=room['Var1'], autopct='%1.1f%%')
    plt.title("Quantidade de quartos, divididos por o tipo dos quartos")
    plt.show()

# Barplot
def bar_plot ():
    plt.figure()
    sns.barplot(data=room, x='Var1', y='Freq')
    plt.show()


# -------------------------------
# Neighbourhood plot
# -------------------------------

def vizinhanca ():
    neig = df['neighbourhood_group'].value_counts().reset_index()
    neig.columns = ['Var1', 'Freq']

    plt.figure()
    sns.barplot(data=neig, x='Freq', y='Var1')
    plt.title("Grupos da vizinhança")
    plt.xlabel("Quantidade")
    plt.ylabel("Grupos da vizinhança")
    plt.show()


# -------------------------------
# Map: Airbnb density in Valencia
# -------------------------------

def mapa ():
    # Load Spain municipalities
    spain = gpd.read_file(
        "https://gist.githubusercontent.com/carlostxm/2f2382656751ae6d85fc35fff118b921/raw/11dd9cc5e122d315b921618f3ce08b46c646df17/spain-municipalities.json"
    )

    # Filter Valencia
    valencia_city = spain[spain['id'] == 46250]

    # Convert Airbnb data to GeoDataFrame
    airbnb_sf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
        crs="EPSG:4326"
    )

    # Load neighbourhoods
    neigh = gpd.read_file("barris-barrios.geojson")

    # Spatial join
    airbnb_neigh = gpd.sjoin(airbnb_sf, neigh, how="left", predicate="intersects")

    # Count per neighbourhood
    counts = airbnb_neigh.groupby('neighbourhood').size().reset_index(name='n')

    # Merge with map
    neigh_map = neigh.merge(counts, left_on='nombre', right_on='neighbourhood', how='left')

    # Plot map
    fig, ax = plt.subplots()
    neigh_map.plot(
        column='n',
        cmap='YlOrRd',
        legend=True,
        edgecolor='black',
        ax=ax
    )
    ax.set_title("Densidade de Airbnbs em Valência")
    ax.axis('off')
    plt.show()


# -------------------------------
# Scatter plots: price relationships
# -------------------------------

def pr_num_rev ():
    plt.figure()
    plt.scatter(df['price'], df['number_of_reviews'])
    plt.xlabel("Preço")
    plt.ylabel("Numero de reviews")
    plt.title("Numero de reviews VS Preço")
    plt.show()

def pr_rev_m ():
    plt.figure()
    plt.scatter(df['price'], df['reviews_per_month'])
    plt.xlabel("Preço")
    plt.ylabel("Reviews por mês")
    plt.title("Numero de reviews mensais VS Preço")
    plt.show()

def pr_rev_y ():
    plt.figure()
    plt.scatter(df['price'], df['number_of_reviews_ltm'])
    plt.xlabel("Preço")
    plt.ylabel("Reviews por ano")
    plt.title("Numero de reviews anuais VS Preço")
    plt.show()

def pr_ava ():
    plt.figure()
    plt.scatter(df['price'], df['availability_365'])
    plt.xlabel("Preço")
    plt.ylabel("Disponibilidade anual")
    plt.title("Disponibilidade anual VS Preço")
    plt.show()



# -------------------------------
# Menu
# -------------------------------
def Op():
    print("\n")
    print("Que gráfico deseja?")
    print("0 - Sair")
    print("1 - Gráficos sobre o tipo de quartos")
    print("2 - Gráficos sobre a vizinhança")
    print("3 - Mapa da densidade de Airbnb em Valencia")
    print("4 - Gráficos de preços")
    print("Que gráfico quer?")
    x = int(input())
    return x

def Op_quartos():
    print("\n")
    print("0 - Sair")
    print("1 - Gráfico circular")
    print("2 - Graffico de barras")
    print("Que gráfico quer?")
    x = int(input())
    return x

def Op_price():
    print("\n")
    print("0 - Sair")
    print("1 - Preço com Número de reviews")
    print("2 - Preço com as Reviews mensais")
    print("3 - Preço com as Reviews Anuais")
    print("4 - Preço com a Disponibilidade anual")
    print("Que gráfico quer?")
    x = int(input())
    return x





Que gráfico deseja?
0 - Sair
1 - Gráficos sobre o tipo de quartos
2 - Gráficos sobre a vizinhança
3 - Mapa da densidade de Airbnb em Valencia
4 - Gráficos de preços
Que gráfico quer?


ValueError: invalid literal for int() with base 10: ''

### Verificar qualidade dos dados

# Data preparation

### Limpeza de dados

#### missings

### Feature engineering

### Selecionar Variáveis

# Modelação dos dados

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# usar neighbourhood como preditor para price (modelo linear simples com dummies)
df_reg = df[['neighbourhood', 'price']].dropna()

X = pd.get_dummies(df_reg['neighbourhood'], drop_first=True)
y = df_reg['price']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R2:", r2_score(y_test, y_pred).round(4))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)).round(4))

coef = pd.Series(model.coef_, index=X.columns)
coef = coef.sort_values(ascending=False)
print("Intercept:", model.intercept_.round(4))
print("Melhores bairros (coef > 0):")
print(coef[coef > 0].head(10))
print("Piores bairros (coef < 0):")
print(coef[coef < 0].head(10))

AttributeError: 'float' object has no attribute 'round'